# L6a: Duality and Sensitivity in Linear Programming

> **Learning objectives**
>
> - Construct the dual interpretation of a resource-allocation LP.
> - Interpret a dual variable as a local marginal value.
> - Check a shadow-price prediction by perturbing a resource bound.
> - Distinguish local sensitivity from a globally valid linear rule.


## Setup

The local setup delegates to the pinned root Julia environment.


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## Motivation
Studying the dual matters because it turns every constraint into a __price__ (called a __shadow value__) which tells us how much an extra unit of any resource would improve the objective. 

> __Dual problem__
>
> __Solution bounds and sensitivity:__ The dual gives tight, computable bounds on the primal problem. Strong duality lets us certify optimality without guesswork, and it exposes sensitivity: which constraints are binding (e.g., at equality), which are slack (non-binding), and how the optimum moves if data changes.
> 
> **Shadow prices (the value of loosening a limit):** Each dual variable tells you how much the best achievable objective would improve if you had one more unit of that specific constrained resource (zero if the constraint is not binding).

> __Algorithms:__ Many algorithms (e.g., simplex and interior-point methods) naturally work with the dual, so understanding it isn’t extra theory; simplex reads the dual to decide pivots while an interior-point method solves the primal and dual in lockstep, using the duality gap to guide and certify convergence.

__TL;DR__ The dual turns each constraint into a __price__, giving provable limits on the best value you can achieve and showing how the optimum would change if you relaxed any constraint. The key insight: the dual tells you how much the objective would improve per unit of any scarce resource (constraint).
___


##  Dual Linear Programming Problems
Having defined primal linear programs, we now turn to their duals, alternative formulations that offer a different perspective on the same optimization problem. You can think of this as viewing the primal through a different lens.

If the __primal problem__ has the form (maximization form):
$$
\begin{align*}
\text{maximize} &\, \sum_{i=1}^{n} c_{i} x_{i}\\
\text{subject to}~\sum_{i=1}^{n} A_{j,i} x_{i} &\leq b_{j}\quad j=1,2,\dots,m\\
x_{i}&\geq 0\qquad i=1,2,\dots,n
\end{align*}
$$
then the __dual problem__ has the form:
$$
\begin{aligned}
\text{minimize}\quad & \sum_{j=1}^{m} b_{j} y_{j}\\
\text{subject to}\quad & \sum_{j=1}^{m} A_{j,i} y_{j} \ge c_{i}
\quad&&i=1,2,\dots,n,\\
&y_{j} \ge 0
\quad&&j=1,2,\dots,m.
\end{aligned}
$$

### What has changed in the dual problem (maximization form)?
There are several key differences between the primal and dual linear programming problems:
1. The objective function flips from maximization to minimization.  
2. Primal objective coefficients $\mathbf{c}_i$ become the dual right-hand side constants.  
3. Primal right-hand side constants $\mathbf{b}_j$ become the dual objective coefficients.  
4. The $m\times n$ constraint matrix $\mathbf{A}$ is transposed in the dual.  
5. The number of variables and constraints swap: the primal has $n$ variables and $m$ constraints, while the dual has $m$ variables and $n$ constraints.
6. Each primal constraint corresponds to a dual variable $\mathbf{y}_j$, and each primal variable $\mathbf{x}_i$ corresponds to a dual constraint.  
7. Inequality directions are inverted in the constraints: a $\le$ constraint in the primal gives rise to a $\ge$ constraint in the dual.   
8. Equality constraints in the primal become free variables in the dual; that is, an equality constraint gives rise to a dual variable $\mathbf{y}_j$ that is free (no sign restriction), while a dual equality constraint gives rise to a primal variable $\mathbf{x}_i$ that is free.

___


## Minimization Form
Above, we presented the primal problem in maximization form. However, it is common to present linear programming problems in minimization form. The primal problem in minimization form is:
$$
\begin{align*}
\text{minimize} &\, \sum_{i=1}^{n} c_{i} x_{i}\\
\text{subject to}~\sum_{i=1}^{n} A_{j,i} x_{i} &\geq b_{j}\quad j=1,2,\dots,m\\
x_{i}&\geq 0\qquad i=1,2,\dots,n
\end{align*}
$$
and the dual problem in this case is:
$$
\begin{aligned}
\text{maximize}\quad & \sum_{j=1}^{m} b_{j} y_{j}\\
\text{subject to}\quad & \sum_{j=1}^{m} A_{j,i} y_{j} \le c_{i}
\quad&&i=1,2,\dots,n,\\
&y_{j} \ge 0
\quad&&j=1,2,\dots,m.
\end{aligned}
$$

### What has changed in the dual problem (minimization form)?
There are several key differences between the primal and dual linear programming problems in minimization form:
1. The objective function flips from minimization to maximization.  
2. Primal objective coefficients $\mathbf{c}_i$ become the dual right-hand side constants.  
3. Primal right-hand side constants $\mathbf{b}_j$ become the dual objective coefficients.  
4. The $m\times n$ constraint matrix $\mathbf{A}$ is transposed in the dual.  
5. The number of variables and constraints swap: the primal has $n$ variables and $m$ constraints, while the dual has $m$ variables and $n$ constraints.
6. Each primal constraint corresponds to a dual variable $\mathbf{y}_j$, and each primal variable $\mathbf{x}_i$ corresponds to a dual constraint.  
7. Inequality directions are inverted in the constraints: a $\ge$ constraint in the primal gives rise to a $\le$ constraint in the dual.   
8. Equality constraints in the primal become free variables in the dual; that is, an equality constraint gives rise to a dual variable $\mathbf{y}_j$ that is free (no sign restriction), while a dual equality constraint gives rise to a primal variable $\mathbf{x}_i$ that is free.

___


## Duality Theory
The solutions of primal and dual linear programming problems are fundamentally connected through duality theory. This relationship provides both theoretical insights and practical computational advantages.

For a primal problem $\max\{\mathbf{c}^{\top} \mathbf{x} : \mathbf{A} \mathbf{x} \le \mathbf{b}, \mathbf{x}\ge\mathbf{0}\}$ and its corresponding dual problem $\min\{\mathbf{b}^{\top} \mathbf{y} : \mathbf{A}^{\top} \mathbf{y} \ge \mathbf{c}, \mathbf{y}\ge\mathbf{0}\}$, the solutions exhibit the following important properties:

> __Weak Duality__ 
> 
> For any primal feasible solution $\mathbf{x}$ and any dual feasible solution $\mathbf{y}$, we have (for maximization primal problems):
> $$
    \begin{align*}
    \mathbf{c}^{\top} \mathbf{x} \le \mathbf{b}^{\top} \mathbf{y}
    \end{align*}
$$
> or equivalently (for minimization primal problems):
> $$
    \begin{align*}
    \mathbf{c}^{\top} \mathbf{x} \ge \mathbf{b}^{\top} \mathbf{y}
    \end{align*}
$$
> This means that the primal objective value is always bounded from above by the dual objective value. 
> The difference $\mathbf{b}^{\top} \mathbf{y} - \mathbf{c}^{\top} \mathbf{x} \geq 0$ is called the **duality gap (maximization)**. 
> Alternatively, for __minimization__ problems, the duality gap is $\mathbf{c}^{\top} \mathbf{x} - \mathbf{b}^{\top} \mathbf{y} \geq 0$.

> __Strong Duality__ 
> 
> If both the primal and dual problems are feasible and have finite optimal values, then:
> $$
\max\{\mathbf{c}^{\top} \mathbf{x}\} = \min\{\mathbf{b}^{\top} \mathbf{y}\}
$$
> In other words, the optimal values of the primal and dual problems are equal (the duality gap is zero). This result means that solving either the primal or dual problem gives us the solution to both.

### Practical Implications
- If we find feasible solutions to both primal and dual problems with equal objective values, we know both are optimal.
- The dual problem can sometimes be easier to solve than the primal, providing an alternative computational approach.
- Sensitivity analysis and economic interpretation often rely on dual variable values (shadow prices).


Let's look at an example to illustrate these concepts.

> __Example__
> 
> [▶ Dual of the Min Cost Max Flow Problem](CHEME-5800-L6a-Example-LP-MinCostMaxFlow-Dual-Fall-2026.ipynb). In this example, we will explore the dual formulation of the minimum cost maximum flow problem that we explored in lab. We will formulate the dual linear programming problem and interpret the dual variables (shadow prices) in the context of network flows.
___


## Lab
In lab L6d, we'll do a fun example where we explore the primal and the dual of an important network flow problem, Flux Balance Analysis (FBA). See you there!


## Summary
In this lecture, we explored the fundamental concepts of dual linear programming and the powerful theoretical relationships between primal and dual problems. We learned how to systematically construct dual formulations and understand their economic significance.

> **Key takeaways:**
>
> * **Dual transformation follows systematic rules.** We learned the mechanical process of converting primal problems to dual problems: objective coefficients become right-hand sides, constraint matrices are transposed, inequality directions flip, and the optimization sense reverses (max↔min). These transformations work consistently for both maximization and minimization primal forms.
> * **Duality theory provides powerful bounds and certificates.** Weak duality guarantees that any primal feasible solution provides a lower bound for any dual feasible solution (for maximization problems), while strong duality ensures that optimal values are equal when both problems are feasible, giving us the fundamental result that solving either problem solves both.
> * **Dual variables have economic interpretation as shadow prices.** The dual variables represent the marginal value of relaxing constraints by one unit, providing insights for sensitivity analysis and resource valuation. This interpretation makes dual problems valuable for understanding the economic structure of optimization problems beyond just finding optimal solutions.

**Where do we go from here?** The systematic dual transformation rules and duality theory we learned provide the foundation for understanding the economic meaning of optimization problems. 

In the upcoming lab, we'll see how dual variables can be interpreted as shadow prices in practical applications like Flux Balance Analysis.

Understanding both primal and dual formulations gives you multiple perspectives on the same optimization problem and prepares you for more advanced topics in linear programming!
___


## Executable sensitivity check


In [2]:
baseline = solve_resource_lp([100.0, 90.0])
resource_one_plus_one = solve_resource_lp([101.0, 90.0])
resource_two_plus_one = solve_resource_lp([100.0, 91.0])

sensitivity = DataFrame(
    resource = ["resource 1", "resource 2"],
    shadow_price = baseline.shadow_prices,
    observed_objective_change = [
        resource_one_plus_one.objective - baseline.objective,
        resource_two_plus_one.objective - baseline.objective,
    ],
)
pretty_table(sensitivity)


┌────────────┬──────────────┬───────────────────────────┐
│   resource │ shadow_price │ observed_objective_change │
│     String │      Float64 │                   Float64 │
├────────────┼──────────────┼───────────────────────────┤
│ resource 1 │          0.8 │                       0.8 │
│ resource 2 │          1.4 │                       1.4 │
└────────────┴──────────────┴───────────────────────────┘


In [3]:
@test baseline.decisions ≈ [42.0, 16.0]
@test baseline.shadow_prices ≈ [0.8, 1.4]
@test sensitivity.observed_objective_change ≈ sensitivity.shadow_price
:duality_sensitivity_verified


:duality_sensitivity_verified

## Summary

The dual prices predict the objective improvement from one additional unit of each binding resource. That prediction is local: a large capacity change can alter the active constraints and therefore the applicable shadow prices.
